# Model Selection and Storage

In this notebook, we'll select models from Hugging Face and save them directly to S3 for our optimization experiments. We'll focus on transformer models that are commonly used in production environments.

## 1. Import Dependencies

In [ ]:
import os
import torch
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import AutoModelForQuestionAnswering, AutoModelForMaskedLM
import boto3
import time
from pathlib import Path

# Import workshop configuration
from workshop_config import S3_BUCKET, AWS_REGION

## 2. Define Models to Download

We'll download several models for different tasks to demonstrate optimization techniques across various model architectures and sizes.

In [ ]:
# Define models to download
models_to_download = {
    "sentiment_analysis": {
        "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
        "task": "sequence-classification",
        "description": "DistilBERT model fine-tuned for sentiment analysis"
    },
    "ner": {
        "model_name": "dbmdz/bert-large-cased-finetuned-conll03-english",
        "task": "token-classification",
        "description": "BERT model fine-tuned for named entity recognition"
    },
    "question_answering": {
        "model_name": "distilbert-base-cased-distilled-squad",
        "task": "question-answering",
        "description": "DistilBERT model fine-tuned for question answering"
    },
    "masked_lm": {
        "model_name": "bert-base-uncased",
        "task": "masked-lm",
        "description": "BERT model for masked language modeling"
    }
}

## 3. Create Function to Download Models and Save to S3

In [ ]:
def download_and_save_to_s3(model_info, bucket_name, temp_dir="temp_models"):
    """Download model from Hugging Face and save directly to S3."""
    model_name = model_info["model_name"]
    task = model_info["task"]
    
    # Create temporary directory if it doesn't exist
    model_dir = os.path.join(temp_dir, model_name.replace("/", "_"))
    os.makedirs(model_dir, exist_ok=True)
    
    print(f"Downloading {model_name} for {task}...")
    
    # Download tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.save_pretrained(model_dir)
    
    # Download model based on task
    if task == "sequence-classification":
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
    elif task == "token-classification":
        model = AutoModelForTokenClassification.from_pretrained(model_name)
    elif task == "question-answering":
        model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    elif task == "masked-lm":
        model = AutoModelForMaskedLM.from_pretrained(model_name)
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Save model to temporary directory
    model.save_pretrained(model_dir)
    
    # Get model size
    model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 * 1024)
    
    # Upload to S3
    s3_client = boto3.client('s3')
    s3_prefix = f"models/{model_name.replace('/', '_')}/"
    
    print(f"Uploading {model_name} to S3...")
    for root, _, files in os.walk(model_dir):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, model_dir)
            s3_key = s3_prefix + relative_path
            s3_client.upload_file(local_path, bucket_name, s3_key)
    
    # Create S3 URI for the model
    s3_uri = f"s3://{bucket_name}/{s3_prefix}"
    
    print(f"Uploaded {model_name} ({model_size_mb:.2f} MB) to {s3_uri}")
    
    return {
        "model_name": model_name,
        "task": task,
        "s3_uri": s3_uri,
        "local_path": model_dir,  # Keep local path for reference
        "size_mb": model_size_mb
    }

## 4. Download Models and Save to S3

In [ ]:
# Create temporary directory
temp_dir = "temp_models"
os.makedirs(temp_dir, exist_ok=True)

# Download models and save to S3
downloaded_models = {}
for key, model_info in models_to_download.items():
    downloaded_models[key] = download_and_save_to_s3(model_info, S3_BUCKET, temp_dir)
    
    # Add a small delay between downloads to avoid rate limiting
    time.sleep(2)

## 5. Create Function to Load Models from S3

In [ ]:
def load_model_from_s3(model_info, temp_dir="temp_models"):
    """Load a model from S3 to local storage for use in the notebook."""
    model_name = model_info["model_name"]
    s3_uri = model_info["s3_uri"]
    local_path = model_info["local_path"]
    
    # Check if model is already downloaded
    if os.path.exists(local_path) and os.path.isfile(os.path.join(local_path, "pytorch_model.bin")):
        print(f"Model {model_name} already exists locally at {local_path}")
        return local_path
    
    print(f"Loading model {model_name} from {s3_uri}...")
    
    # Parse S3 URI
    s3_parts = s3_uri.replace("s3://", "").split("/")
    bucket = s3_parts[0]
    prefix = "/".join(s3_parts[1:])
    
    # Create S3 client
    s3_client = boto3.client('s3')
    
    # List objects in the prefix
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    
    # Download each file
    os.makedirs(local_path, exist_ok=True)
    for obj in response.get('Contents', []):
        key = obj['Key']
        filename = os.path.basename(key)
        if filename:  # Skip directory entries
            local_file = os.path.join(local_path, filename)
            s3_client.download_file(bucket, key, local_file)
    
    print(f"Model loaded to {local_path}")
    return local_path

## 6. Test Loading a Model from S3

In [ ]:
# Test loading a model from S3
model_key = "sentiment_analysis"  # Choose one model to test
model_info = downloaded_models[model_key]

local_path = load_model_from_s3(model_info)
print(f"Model loaded to {local_path}")

# Test loading the model with transformers
tokenizer = AutoTokenizer.from_pretrained(local_path)
task = model_info["task"]

if task == "sequence-classification":
    model = AutoModelForSequenceClassification.from_pretrained(local_path)
elif task == "token-classification":
    model = AutoModelForTokenClassification.from_pretrained(local_path)
elif task == "question-answering":
    model = AutoModelForQuestionAnswering.from_pretrained(local_path)
elif task == "masked-lm":
    model = AutoModelForMaskedLM.from_pretrained(local_path)

print(f"Successfully loaded model and tokenizer from {local_path}")

## 7. Save Model Information

In [ ]:
# Save model information to file
with open('model_info.json', 'w') as f:
    json.dump(downloaded_models, f, indent=2)

print("Model information saved to model_info.json")

## 8. Display Model Summary

In [ ]:
# Create a DataFrame with model information
model_data = []
for model_key, model_info in downloaded_models.items():
    model_data.append({
        "Task": model_key,
        "Model": model_info["model_name"],
        "Size (MB)": f"{model_info['size_mb']:.2f}",
        "S3 Location": model_info["s3_uri"]
    })

# Display as a table
pd.DataFrame(model_data)

## 9. Clean Up Temporary Files (Optional)

In [ ]:
# Uncomment to remove temporary files
# import shutil
# shutil.rmtree(temp_dir)
# print(f"Removed temporary directory: {temp_dir}")

## 10. Next Steps

Now that we've selected and saved our models to S3, we're ready to proceed to the next notebook where we'll establish baseline performance metrics for each model.